In [22]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [23]:
PROJECT_ROOT = r"D:\Finalyear project 2\fyp-2"   # 🔴 adjust only if needed
os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())


Working directory: D:\Finalyear project 2\fyp-2


In [24]:
assert os.path.exists("data/splits/patient_splits.json")
print("Split file found ✔️")


Split file found ✔️


In [25]:
from src.models.attention_fusion import AttentionFusionModel
from src.models.attention_dataset import AttentionFusionDataset
from src.models.collate import collate_attention


In [26]:
train_dataset = AttentionFusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="train",
    label_map={"dementia": 1, "no_dementia": 0}
)

val_dataset = AttentionFusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="val",
    label_map={"dementia": 1, "no_dementia": 0}
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 249
Val samples: 54


In [27]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,              # ⚠️ Batch size 1 for 4GB GPU (most stable)
    shuffle=True,
    collate_fn=collate_attention
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_attention
)

print(f"✓ DataLoaders created")
print(f"  - Train batches: {len(train_loader)} (batch size: 1)")
print(f"  - Val batches: {len(val_loader)} (batch size: 1)")
print(f"  - Total train samples: {len(train_dataset)}")
print(f"  - Total val samples: {len(val_dataset)}")

✓ DataLoaders created
  - Train batches: 249 (batch size: 1)
  - Val batches: 54 (batch size: 1)
  - Total train samples: 249
  - Total val samples: 54


In [28]:
model = AttentionFusionModel(dim=768).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

# Enhanced optimizer with weight decay (L2 regularization)
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=1e-3,           # Slightly higher for simpler model
    weight_decay=1e-4,  # L2 regularization
    betas=(0.9, 0.999)
)

# Learning rate scheduler for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,           # Restart every 10 epochs
    T_mult=2,         # Double the period each restart
    eta_min=1e-6      # Minimum learning rate
)

print("✓ Model initialized (BASELINE - 4GB GPU optimized)")
print(f"  - Architecture: Simple feature fusion (no attention)")
print(f"  - Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  - Device: {DEVICE}")
print(f"  - Dropout: Aggressive (0.4 → 0.3 → 0.2)")
print(f"  - Batch Size: 1 (most stable for 4GB GPU)")
print(f"  - Learning Rate: 1e-3 (adapted for simpler model)")

# Memory info
if DEVICE == "cuda":
    print(f"  - GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

model

✓ Model initialized (BASELINE - 4GB GPU optimized)
  - Architecture: Simple feature fusion (no attention)
  - Parameters: 25,768,705
  - Device: cuda
  - Dropout: Aggressive (0.4 → 0.3 → 0.2)
  - Batch Size: 1 (most stable for 4GB GPU)
  - Learning Rate: 1e-3 (adapted for simpler model)
  - GPU Memory: 4.3GB


AttentionFusionModel(
  (audio_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (text_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (audio_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (audio_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (text_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (text_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (audio_to_text_attentions): ModuleList(
    (0-1): 2 x MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  

In [31]:

model.eval()   # IMPORTANT

AttentionFusionModel(
  (audio_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (text_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (audio_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (audio_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (text_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (text_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (audio_to_text_attentions): ModuleList(
    (0-1): 2 x MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  

In [32]:
# Memory usage test before training
print("Verifying memory requirements...\n")

if DEVICE == "cuda":
    # Clear cache
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    initial_memory = torch.cuda.memory_allocated() / 1e9
    print(f"Initial GPU Memory: {initial_memory:.3f}GB")
    
    # Test forward + backward pass
    try:
        test_audio, test_text, test_y = next(iter(train_loader))
        test_audio = test_audio.to(DEVICE)
        test_text = test_text.to(DEVICE)
        test_y = test_y.to(DEVICE).float()
        
        # Forward pass
        test_logits = model(test_audio, test_text)
        test_loss = criterion(test_logits, test_y)
        
        # Backward pass
        optimizer.zero_grad()
        test_loss.backward()
        optimizer.step()
        
        torch.cuda.empty_cache()
        
        peak_memory = torch.cuda.max_memory_allocated() / 1e9
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        print(f"Peak Memory (train step): {peak_memory:.3f}GB / {total_memory:.1f}GB")
        print(f"Memory Margin: {(total_memory - peak_memory):.3f}GB available")
        
        if peak_memory < total_memory * 0.8:
            print("\n✅ SAFE TO TRAIN - Memory test passed!")
        else:
            print("\n⚠️  WARNING - Limited memory margin (consider CPU training)")
    except RuntimeError as e:
        print(f"❌ FAILED - {str(e)}")
        print("Switch to CPU training or reduce model size")
else:
    print("✅ Using CPU - no GPU memory constraints\n")

Verifying memory requirements...

Initial GPU Memory: 1.623GB
Peak Memory (train step): 3.198GB / 4.3GB
Memory Margin: 1.097GB available

✅ SAFE TO TRAIN - Memory test passed!


In [33]:
# Display model architecture
print("\n" + "="*60)
print("MODEL ARCHITECTURE")
print("="*60)
print(model)
print("\n" + "="*60)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Model Size: {(total_params * 4) / (1024**2):.2f} MB")
print("="*60 + "\n")


MODEL ARCHITECTURE
AttentionFusionModel(
  (audio_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (text_proj): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (audio_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (audio_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (text_self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (text_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (audio_to_text_attentions): ModuleList(
    (0-1): 2 x MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_feature

In [34]:
def evaluate(model, loader, criterion=None, device="cuda"):
    """Enhanced evaluation with loss tracking"""
    model.eval()
    preds, labels = [], []
    val_loss = 0.0

    with torch.no_grad():
        for audio, text, y in loader:
            audio = audio.to(device)
            text = text.to(device)
            y = y.to(device).float()

            logits = model(audio, text)
            probs = torch.sigmoid(logits)

            # Track loss if criterion provided
            if criterion is not None:
                loss = criterion(logits, y)
                val_loss += loss.item()

            pred = (probs > 0.5).long()
            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    
    if criterion is not None:
        val_loss /= len(loader)
        return acc, f1, val_loss, preds, labels
    
    return acc, f1, preds, labels

In [35]:
# Enhanced training with early stopping and checkpointing
# For better gradient estimates with batch_size=1, accumulate over multiple steps
EPOCHS = 100  # More epochs since batch_size=1 gives noisy gradients
PATIENCE = 15  # Slightly higher patience for noisier gradients
ACCUMULATION_STEPS = 4  # Accumulate gradients over 4 steps (effective batch size ~4)
best_f1 = 0.0
patience_counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
    "learning_rate": []
}

print("Training with gradient accumulation and early stopping...\n")
print("⚠️  BASELINE MODEL (Memory Optimized):")
print("   - Batch size: 1 (very safe)")
print("   - Gradient accumulation: 4 steps (effective batch ~4)")
print("   - Model: No attention (simple fusion)")
print("   - Epochs: 100 (more needed for batch_size=1)\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    accumulation_counter = 0

    for batch_idx, (audio, text, y) in enumerate(train_loader):
        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        y = y.to(DEVICE).float()

        # Forward pass
        logits = model(audio, text)
        loss = criterion(logits, y) / ACCUMULATION_STEPS  # Scale loss for accumulation
        loss.backward()
        
        accumulation_counter += 1
        running_loss += loss.item() * ACCUMULATION_STEPS

        # Update weights every ACCUMULATION_STEPS
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            optimizer.zero_grad()
            
            # Clear GPU cache periodically
            if DEVICE == "cuda" and (batch_idx + 1) % (ACCUMULATION_STEPS * 4) == 0:
                torch.cuda.empty_cache()

    avg_train_loss = running_loss / len(train_loader)
    
    # Validation
    val_acc, val_f1, val_loss, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    
    # Learning rate scheduling
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    history["learning_rate"].append(current_lr)

    # Print progress (every 5 epochs to reduce output)
    if epoch % 5 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.3f} | "
            f"Val F1: {val_f1:.3f} | "
            f"LR: {current_lr:.2e}"
        )

    # Checkpointing: save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "attention_fusion_model_best.pt")
        if epoch % 5 == 0 or val_f1 > best_f1 * 0.99:  # Only print on improvement
            print(f"  → 🔥 Best F1: {val_f1:.4f}! Saved.")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n✓ Early stopping at epoch {epoch}")
        print(f"  - Best F1 score: {best_f1:.4f}")
        break

print(f"\n✓ Training complete!")
print(f"  - Epochs trained: {len(history['train_loss'])}")
print(f"  - Best F1: {best_f1:.4f}")
print(f"  - Best Val Accuracy: {history['val_acc'][history['val_f1'].index(best_f1)]:.4f}")

Training with gradient accumulation and early stopping...

⚠️  BASELINE MODEL (Memory Optimized):
   - Batch size: 1 (very safe)
   - Gradient accumulation: 4 steps (effective batch ~4)
   - Model: No attention (simple fusion)
   - Epochs: 100 (more needed for batch_size=1)



ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 512])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Find best epoch
best_epoch = np.argmax(history["val_f1"])
print(f"📊 BEST PERFORMANCE SUMMARY")
print(f"  - Best Epoch: {best_epoch + 1}")
print(f"  - Best Val Accuracy: {history['val_acc'][best_epoch]:.4f}")
print(f"  - Best Val F1: {history['val_f1'][best_epoch]:.4f}")
print(f"  - Best Val Loss: {history['val_loss'][best_epoch]:.4f}")
print(f"  - Training Loss at Best: {history['train_loss'][best_epoch]:.4f}")

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curves
ax = axes[0, 0]
ax.plot(history["train_loss"], label="Train Loss", marker='o', markersize=4)
ax.plot(history["val_loss"], label="Val Loss", marker='s', markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy curve
ax = axes[0, 1]
ax.plot(history["val_acc"], label="Val Accuracy", marker='o', markersize=4, color='green')
ax.axhline(y=history["val_acc"][best_epoch], color='r', linestyle='--', label=f"Best: {history['val_acc'][best_epoch]:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Validation Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# F1 Score curve
ax = axes[1, 0]
ax.plot(history["val_f1"], label="Val F1", marker='o', markersize=4, color='orange')
ax.axhline(y=history["val_f1"][best_epoch], color='r', linestyle='--', label=f"Best: {history['val_f1'][best_epoch]:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1 Score")
ax.set_title("Validation F1 Score")
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[1, 1]
ax.semilogy(history["learning_rate"], marker='o', markersize=4, color='purple')
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate (log scale)")
ax.set_title("Learning Rate Schedule")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Training history saved to 'training_history.png'")

Best Epoch: 16
Best Val Accuracy: 0.7962962962962963
Best Val F1: 0.7555555555555555


In [ ]:
# Load best model checkpoint for final evaluation
model.load_state_dict(torch.load("attention_fusion_model_best.pt", map_location=DEVICE))
print("✓ Loaded best model checkpoint")

# Evaluate on validation set
val_acc, val_f1, preds, labels = evaluate(model, val_loader, device=DEVICE)

print(f"\n📈 FINAL VALIDATION METRICS")
print(f"  - Accuracy: {val_acc:.4f}")
print(f"  - F1 Score: {val_f1:.4f}")

print("\n📋 CLASSIFICATION REPORT")
print(
    classification_report(
        labels,
        preds,
        target_names=["No Dementia", "Dementia"],
        digits=4
    )
)

              precision    recall  f1-score   support

 No Dementia       0.66      0.94      0.77        31
    Dementia       0.80      0.35      0.48        23

    accuracy                           0.69        54
   macro avg       0.73      0.64      0.63        54
weighted avg       0.72      0.69      0.65        54



In [ ]:
# Save final model
torch.save(model.state_dict(), "attention_fusion_model_final.pt")

print("✓ Models saved:")
print("  1. attention_fusion_model_best.pt (best F1 during training)")
print("  2. attention_fusion_model_final.pt (final model)")

# Display model info
print(f"\n📋 Model Information:")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Parameters: {trainable_params:,}")
print(f"  - Architecture: AttentionFusionModel (Enhanced)")
print(f"  - Attention Heads: 8")
print(f"  - Attention Blocks: 2 (bidirectional)")
print(f"  - Best F1 Score: {best_f1:.4f}")

Attention fusion model saved ✔️


## 🚀 Model: Baseline Classifier (4GB GPU Optimized)

### Architecture
- ✅ **Simple Feature Fusion**: No attention mechanism (too memory-intensive on 4GB GPU)
- ✅ **Mean + Max Pooling**: Each modality pooled independently
- ✅ **Concatenated Features**: [Audio (768) + Text (768)] = 1536
- ✅ **Robust Classifier**: 4-layer network with BatchNorm
- ✅ **Progressive Dropout**: 0.4 → 0.3 → 0.2 (regularization decay)
- ✅ **LayerNorm Stabilization**: Better training dynamics

### Training Strategy
- ✅ **Gradient Accumulation**: 4 steps (effective batch ~4 on batch_size=1)
- ✅ **Learning Rate Scheduling**: CosineAnnealingWarmRestarts for adaptive decay
- ✅ **Early Stopping**: Halts at 15 epochs without improvement
- ✅ **Model Checkpointing**: Auto-saves best F1 model
- ✅ **Gradient Clipping**: Prevents exploding gradients
- ✅ **AdamW Optimizer**: L2 regularization (weight_decay=1e-4)
- ✅ **100 Epochs**: More iterations for batch_size=1 gradient stability
- ✅ **Memory Cleanup**: GPU cache clearing every 4 accumulated steps

### Memory Optimization (4GB GPU)
- ✅ **Batch Size: 1** - No attention means small memory footprint
- ✅ **Gradient Accumulation**: Simulates batch_size=4 without memory overhead
- ✅ **No Attention Mechanism**: Avoids massive Q*K^T matrices
- ✅ **Simple Pooling**: Mean + max (no learned transformations)
- ✅ **Tight Regularization**: Prevents overfitting on small batches

### Hyperparameters
- Learning Rate: 1e-3
- Weight Decay: 1e-4
- Batch Size: 1
- Accumulation Steps: 4
- Epochs: 100
- Patience: 15
- Optimizer: AdamW

### Why Baseline Over Attention?
- 💾 **Memory**: Attention creates O(T²) matrices in VRAM
- 🚀 **Stability**: Large batches improve optimization → use accumulation
- 📊 **Performance**: Proper pooling + dense layers often competitive
- 🔧 **Debugging**: Simpler model ↔ easier hyperparameter tuning
- 📈 **Accuracy**: Can reach 85-95% on dementia detection tasks

## 📝 Next Steps & Improvements

### Currently Running
✅ **Baseline Model** - Simple feature fusion optimized for 4GB GPU
- No memory issues expected
- Gradient accumulation ensures stable training
- Should train successfully to completion

### Performance Optimizations
1. **Hyperparameter Tuning**
   - Learning rate: Try 5e-4, 1e-3, 2e-3
   - Dropout rates: Adjust 0.4/0.3/0.2 based on overfitting
   - Accumulation steps: Try 2, 4, 8

2. **Feature Engineering**
   - Standardize audio/text features before training
   - Compute deltas (temporal derivatives)
   - Try different pooling strategies (mean, max, min)

3. **Data ImprovedmentS**
   - Data augmentation (small perturbations to features)
   - Class balancing if dementia/control imbalanced
   - Ensure features are properly normalized

### Upgrading to Attention (requires more GPU)
If you upgrade to 6GB+ GPU, can enable attention:
```python
# Switch model class:
class AttentionFusionModel(nn.Module):
    def __init__(self, dim=768, heads=4):
        # Add MultiheadAttention layer
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )
```

### If Still Getting OOM on 4GB
1. **Switch to CPU training**: `DEVICE = "cpu"` (much slower)
2. **Reduce batch_size to 0.5**: Process every other sample, repeat
3. **Use mixed precision**:
   ```python
   from torch.cuda.amp import autocast
   with autocast():
       logits = model(audio, text)
   ```
4. **Use activation checkpointing** (saves memory, slower)

### Monitoring Training
- Watch for overfitting: val_loss > train_loss
- Check learning curves for plateaus
- Save best model automatically (done ✓)
- Monitor memory usage each epoch

### After Training
1. Evaluate on test set (separate from val)
2. Generate SHAP explanations (use SHAP_Explainability.ipynb)
3. Check feature importance
4. Validate with domain experts